Homework Reflection 8

1. Include the code you used to solve Week 8's coding quiz problems and write about the obstacles / challenges / insights you encountered while solving them.

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

df = pd.read_csv('/Users/gkumargaur/workspace/datascience/boston_university/semester3/ms_datascience/data/homework_8.1.csv')

# Step 1: propensity model - Z predicts treatment X (C=np.inf -> no L2 shrinkage)
ps_model = LogisticRegression(C=np.inf).fit(df[['Z']], df['X'])
ps_model_default = LogisticRegression().fit(df[['Z']], df['X'])   # comparison: default C=1.0

# Step 2: propensity scores p = P(X=1 | Z)
df['p'] = ps_model.predict_proba(df[['Z']])[:, 1]
p_default = ps_model_default.predict_proba(df[['Z']])[:, 1]

# Step 3: inverse probability weights
df['ipw'] = np.where(df['X'] == 1, 1 / df['p'], 1 / (1 - df['p']))
w_default = np.where(df['X'] == 1, 1 / p_default, 1 / (1 - p_default))

# Step 4: ATE = weighted mean Y (treated) - weighted mean Y (untreated)
treated, control = df[df['X'] == 1], df[df['X'] == 0]
ate_ipw = (np.average(treated['Y'], weights=treated['ipw'])
           - np.average(control['Y'], weights=control['ipw']))
naive = treated['Y'].mean() - control['Y'].mean()

X, Y, p = df['X'], df['Y'], df['p']
ate_ht = np.mean(X * Y / p) - np.mean((1 - X) * Y / (1 - p))     # unnormalized (Horvitz-Thompson)
t, c = (df['X'] == 1).values, (df['X'] == 0).values
ate_default = (np.average(df['Y'][t], weights=w_default[t])
               - np.average(df['Y'][c], weights=w_default[c]))

print(f"Propensity score range: {df['p'].min():.3f} - {df['p'].max():.3f},  max weight: {df['ipw'].max():.1f}")
print(f"Naive difference in means:                {naive:.3f}")
print(f"IPW ATE estimate (main):                  {ate_ipw:.3f}")
print(f"IPW ATE, unnormalized (Horvitz-Thompson): {ate_ht:.3f}")
print(f"IPW ATE, sklearn default C=1.0:           {ate_default:.3f}")

# Q2: propensity scores of the first three rows
print("\nQ2 - first three propensity scores:", df['p'].head(3).round(3).tolist())

Propensity score range: 0.047 - 0.933,  max weight: 12.9
Naive difference in means:                3.037
IPW ATE estimate (main):                  2.268
IPW ATE, unnormalized (Horvitz-Thompson): 2.236
IPW ATE, sklearn default C=1.0:           2.274

Q2 - first three propensity scores: [0.842, 0.585, 0.712]


In [2]:
from scipy.spatial.distance import mahalanobis

df2 = pd.read_csv('/Users/gkumargaur/workspace/datascience/boston_university/semester3/ms_datascience/data/homework_8.2.csv')

# Q3 Step 1: inverse covariance from ALL rows; np.cov needs a 2 x N matrix (rows = variables)
Z = np.vstack((df2['Z1'], df2['Z2']))          # shape (2, 1000)
inv_cov = np.linalg.inv(np.cov(Z))             # shape (2, 2)

# Step 2: split into treated and untreated
treated2 = df2[df2['X'] == 1]
untreated2 = df2[df2['X'] == 0]
T = treated2[['Z1', 'Z2']].values
C = untreated2[['Z1', 'Z2']].values

# Step 3: nearest untreated row for each treated row (with replacement: controls are never removed)
match_idx, match_dist = [], []
for z_t in T:
    dists = [mahalanobis(z_t, z_c, inv_cov) for z_c in C]
    match_idx.append(np.argmin(dists))
    match_dist.append(np.min(dists))           # kept for Q4
match_dist = np.array(match_dist)

# Step 4: ATT = mean(treated Y - matched control Y)
att = np.mean(treated2['Y'].values - untreated2['Y'].values[match_idx])
print(f"Simple difference in means: {treated2['Y'].mean() - untreated2['Y'].mean():.3f}")
print(f"Matched ATT:                {att:.3f}")

# Q4: least common support = treated row whose NEAREST control is farthest away
worst = np.argmax(match_dist)
print(f"\nQ4 - farthest nearest-match distance: {match_dist[worst]:.3f}")
print("Treated item (Z1, Z2):     ", treated2.iloc[worst][['Z1', 'Z2']].values.round(3))
print("Nearest untreated (Z1, Z2):", untreated2.iloc[match_idx[worst]][['Z1', 'Z2']].values.round(3))

Simple difference in means: 4.048
Matched ATT:                3.438

Q4 - farthest nearest-match distance: 1.383
Treated item (Z1, Z2):      [2.696 0.538]
Nearest untreated (Z1, Z2): [ 1.52  -1.282]


**Approach and choices.** For Questions 1–2, I estimated the ATE with inverse probability weighting. A logistic regression of X on Z gave each row's propensity score p = P(X=1 | Z), and each row was weighted by 1/p if treated or 1/(1−p) if untreated. I made three choices. First, I set `C=np.inf` to turn off scikit-learn's default L2 penalty, which shrinks coefficients and pulls the scores toward 0.5; we want the plain probability here. Second, I used normalized (Hájek) weighted means via `np.average` rather than the unnormalized Horvitz-Thompson formula, because normalized weights are more stable when a few weights are large. Third, I checked overlap before trusting the result. The IPW ATE was **≈ 2.27**, compared with a naive difference of 3.04. For Questions 3–4, I matched each treated row to its nearest untreated row by Mahalanobis distance, with replacement, which gave an **ATT ≈ 3.44** compared with a naive 4.05.

**Obstacles.** The main pitfall was the covariance matrix. `np.cov` treats each *row* as a variable, so passing an N × 2 DataFrame silently returns a 1000 × 1000 matrix. I had to stack Z1 and Z2 into a 2 × N array to get the 2 × 2 covariance, then invert it. A second subtle point: "with replacement" needed no extra code. Each treated row searches all controls and nothing is ever removed, so a control can be reused. For Question 4, I first used the overall largest distance, but "least common support" means the treated row whose *nearest* control is farthest away, so I had to store the *minimum* distance per treated row and then take the maximum. Finally, the default regularization was a hidden choice I only noticed when I compared it with the unpenalized fit; it turned out to matter very little (2.274 vs 2.268).

**Insights.** Both methods pulled the estimate down from the naive difference. That's because Z (or Z1 and Z2) raises both the chance of treatment and the outcome, so the treated group would have had higher Y anyway. Reweighting and matching both make the groups comparable on Z, but they answer slightly different questions. IPW with these weights estimates the **ATE** over everyone, while matching treated rows to controls estimates the **ATT**, the effect on those who were actually treated. Checking robustness was useful too: three IPW variants landed between 2.24 and 2.27, which gave me confidence in the result. The overlap check was also reassuring: scores were between 0.05 and 0.93 and the largest weight was about 13, so no single row dominated. Both methods still rely on the same untestable assumption, that the measured Z's capture all the confounding.